# Preprocesamiento recomendado tras el EDA

El EDA no detecta problemas graves de calidad: no hay duplicados completos, las claves primarias son unicas, las foreign keys tienen cobertura del 100%, las tablas resumen cuadran con `daily` y los assets existen. El preprocesamiento necesario es sobre todo estructural:

- tipar fechas y limpiar strings;
- separar dimensiones, facts y targets;
- evitar columnas duplicadas en joins;
- expandir `campaigns.countries` a una tabla normalizada;
- recalcular ventanas temporales desde `daily` y no usar `impressions_last_7d`, porque depende del orden de filas;
- tratar `creative_summary` y `campaign_summary` como agregados/labels, no como features tempranas.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "Smadex_Creative_Intelligence_Dataset_FULL").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "Smadex_Creative_Intelligence_Dataset_FULL"
OUTPUT_DIR = ROOT / "feature" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "advertisers": "advertisers.csv",
    "campaigns": "campaigns.csv",
    "creatives": "creatives.csv",
    "daily": "creative_daily_country_os_stats.csv",
    "creative_summary": "creative_summary.csv",
    "campaign_summary": "campaign_summary.csv",
    "dictionary": "data_dictionary.csv",
}

DATE_COLS = {
    "campaigns": ["start_date", "end_date"],
    "creatives": ["creative_launch_date"],
    "daily": ["date"],
    "creative_summary": ["creative_launch_date"],
    "campaign_summary": ["start_date", "end_date"],
}

DATA_DIR, OUTPUT_DIR


(PosixPath('/home/wei/projects/hack_UPC_2026/Smadex_Creative_Intelligence_Dataset_FULL'),
 PosixPath('/home/wei/projects/hack_UPC_2026/feature/processed'))

## Decision por CSV

| CSV | Hace falta preprocesamiento? | Accion |
|---|---|---|
| `advertisers.csv` | Minimo | Limpiar strings, validar `advertiser_id`, usar solo como dimension para `hq_region`. |
| `campaigns.csv` | Si | Parsear fechas, normalizar `countries`, crear `campaign_dim`, validar fechas y FK. |
| `creatives.csv` | Si | Parsear `creative_launch_date`, validar assets, agregar features estaticas (`aspect_ratio`, `asset_pixels`, `is_video_like`) y unir contexto de campana sin duplicar columnas. |
| `creative_daily_country_os_stats.csv` | Si, importante | Crear fact limpia, eliminar/ignorar `impressions_last_7d`, calcular ratios y agregar a `creative_day` para ventanas temporales correctas. |
| `creative_summary.csv` | Si, por uso | Convertir `fatigue_day` a nullable integer y separar targets/agregados para evitar leakage. |
| `campaign_summary.csv` | Si, por uso | Mantener como KPIs agregados de campana; no usar como dimension base. |
| `data_dictionary.csv` | No | Es metadata de referencia. |


In [2]:
def load_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(DATA_DIR / FILES[name], parse_dates=DATE_COLS.get(name, []))


raw = {name: load_csv(name) for name in FILES}

shape_report = pd.DataFrame(
    [
        {"table": name, "rows": len(df), "columns": df.shape[1], "duplicate_rows": int(df.duplicated().sum())}
        for name, df in raw.items()
    ]
).sort_values("rows", ascending=False)

shape_report


,table,rows,columns,duplicate_rows
3,daily,192315,14,0
4,creative_summary,1080,59,0
2,creatives,1080,32,0
5,campaign_summary,180,22,0
1,campaigns,180,14,0
6,dictionary,145,4,0
0,advertisers,36,4,0


In [3]:
def clean_strings(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.select_dtypes(include=["object", "string"]).columns:
        out[col] = out[col].astype("string").str.strip()
    return out


def safe_div(num, den):
    return np.where(den == 0, np.nan, num / den)


def add_rate_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if {"clicks", "impressions"}.issubset(out.columns):
        out["ctr"] = safe_div(out["clicks"], out["impressions"])
    if {"conversions", "clicks"}.issubset(out.columns):
        out["cvr"] = safe_div(out["conversions"], out["clicks"])
    if {"conversions", "impressions"}.issubset(out.columns):
        out["ipm"] = safe_div(out["conversions"] * 1000, out["impressions"])
    if {"revenue_usd", "spend_usd"}.issubset(out.columns):
        out["roas"] = safe_div(out["revenue_usd"], out["spend_usd"])
    if {"viewable_impressions", "impressions"}.issubset(out.columns):
        out["viewability_rate"] = safe_div(out["viewable_impressions"], out["impressions"])
    if {"video_completions", "impressions"}.issubset(out.columns):
        out["video_completion_rate"] = safe_div(out["video_completions"], out["impressions"])
    return out


def assert_unique(df: pd.DataFrame, cols: list[str], name: str) -> None:
    duplicated = int(df.duplicated(cols).sum())
    if duplicated:
        raise ValueError(f"{name} has {duplicated} duplicated rows on {cols}")


def assert_fk(left: pd.DataFrame, right: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = left[cols].drop_duplicates().merge(right[cols].drop_duplicates(), on=cols, how="left", indicator=True)
    missing = missing[missing["_merge"].eq("left_only")]
    if len(missing):
        raise ValueError(f"{name} has {len(missing)} missing FK values for {cols}")


In [4]:
advertisers = clean_strings(raw["advertisers"])
campaigns = clean_strings(raw["campaigns"])
creatives = clean_strings(raw["creatives"])
daily = clean_strings(raw["daily"])
creative_summary = clean_strings(raw["creative_summary"])
campaign_summary = clean_strings(raw["campaign_summary"])

assert_unique(advertisers, ["advertiser_id"], "advertisers")
assert_unique(campaigns, ["campaign_id"], "campaigns")
assert_unique(creatives, ["creative_id"], "creatives")
assert_unique(daily, ["date", "creative_id", "country", "os"], "daily")
assert_unique(creative_summary, ["creative_id"], "creative_summary")
assert_unique(campaign_summary, ["campaign_id"], "campaign_summary")

assert_fk(campaigns, advertisers, ["advertiser_id"], "campaigns -> advertisers")
assert_fk(creatives, campaigns, ["campaign_id"], "creatives -> campaigns")
assert_fk(daily, campaigns, ["campaign_id"], "daily -> campaigns")
assert_fk(daily, creatives, ["creative_id"], "daily -> creatives")
assert_fk(creative_summary, creatives, ["creative_id"], "creative_summary -> creatives")
assert_fk(campaign_summary, campaigns, ["campaign_id"], "campaign_summary -> campaigns")

"Validaciones base OK"


'Validaciones base OK'

## 1. Advertisers y campaigns

`advertisers.csv` no necesita limpieza real. `campaigns.csv` si necesita dos preparaciones: fechas como fechas y `countries` normalizado en una tabla `campaign_country`, porque es una lista separada por pipes.

In [5]:
advertisers_clean = advertisers.copy()

campaigns_clean = campaigns.copy()
campaigns_clean["campaign_duration_days"] = (campaigns_clean["end_date"] - campaigns_clean["start_date"]).dt.days + 1
if campaigns_clean["campaign_duration_days"].le(0).any():
    raise ValueError("Hay campanas con end_date anterior a start_date")

campaign_country = (
    campaigns_clean[["campaign_id", "countries"]]
    .assign(country=lambda df: df["countries"].str.split("|"))
    .explode("country")
    .drop(columns="countries")
    .reset_index(drop=True)
)

campaign_dim = (
    campaigns_clean.drop(columns=["countries"])
    .merge(advertisers_clean[["advertiser_id", "hq_region"]], on="advertiser_id", how="left", validate="m:1")
)

campaign_dim.shape, campaign_country.shape


((180, 15), (502, 2))

## 2. Creatives

`creatives.csv` es la dimension principal para metadata creativa. Se conserva `asset_file` como path, se valida que el fichero existe y se agregan features estaticas simples. El contexto de campana se agrega desde `campaign_dim` sin traer columnas duplicadas que ya existen en creatives.

In [6]:
creative_dim = creatives.copy()
creative_dim["asset_path"] = creative_dim["asset_file"].map(lambda p: str(DATA_DIR / p))
creative_dim["asset_exists"] = creative_dim["asset_path"].map(lambda p: Path(p).exists())
creative_dim["aspect_ratio"] = safe_div(creative_dim["width"], creative_dim["height"])
creative_dim["asset_pixels"] = creative_dim["width"] * creative_dim["height"]
creative_dim["is_video_like"] = creative_dim["duration_sec"].gt(0).astype("int8")
creative_dim["headline_chars"] = creative_dim["headline"].str.len()
creative_dim["subhead_chars"] = creative_dim["subhead"].str.len()

if not creative_dim["asset_exists"].all():
    raise ValueError("Hay assets referenciados que no existen")

campaign_context_cols = [
    "campaign_id",
    "advertiser_id",
    "objective",
    "primary_theme",
    "target_age_segment",
    "target_os",
    "start_date",
    "end_date",
    "daily_budget_usd",
    "kpi_goal",
    "campaign_duration_days",
    "hq_region",
]

creative_dim = creative_dim.merge(
    campaign_dim[campaign_context_cols], on="campaign_id", how="left", validate="m:1"
)

creative_dim.shape


(1080, 50)

## 3. Daily fact y creative_day

La tabla diaria conserva su grano original `date x creative_id x country x os`. Para modelado temporal se crea tambien `creative_day`, una version agregada a `creative_id x date`. Se elimina `impressions_last_7d` de las features porque el EDA confirma que es rolling de 7 filas anteriores, no de 7 dias calendario.

In [7]:
daily_clean = daily.sort_values(["creative_id", "date", "country", "os"]).reset_index(drop=True)
daily_clean = daily_clean.drop(columns=["impressions_last_7d"])
daily_clean = add_rate_columns(daily_clean)

daily_enriched = daily_clean.merge(
    creative_dim,
    on=["creative_id", "campaign_id"],
    how="left",
    validate="m:1",
)

creative_day = (
    daily_clean.groupby(["creative_id", "campaign_id", "date", "days_since_launch"], as_index=False)
    .agg(
        spend_usd=("spend_usd", "sum"),
        impressions=("impressions", "sum"),
        viewable_impressions=("viewable_impressions", "sum"),
        clicks=("clicks", "sum"),
        conversions=("conversions", "sum"),
        revenue_usd=("revenue_usd", "sum"),
        video_completions=("video_completions", "sum"),
        country_os_slices=("country", "size"),
    )
    .sort_values(["creative_id", "date"])
    .reset_index(drop=True)
)
creative_day = add_rate_columns(creative_day)

daily_enriched.shape, creative_day.shape


((192315, 67), (52349, 18))

In [8]:
ROLLING_METRICS = ["impressions", "clicks", "conversions", "spend_usd", "revenue_usd"]

for window in [3, 7, 14]:
    for metric in ROLLING_METRICS:
        col = f"{metric}_prev_{window}d"
        creative_day[col] = (
            creative_day.groupby("creative_id")[metric]
            .transform(lambda s: s.shift(1).rolling(window, min_periods=1).sum())
            .fillna(0)
        )

    creative_day[f"ctr_prev_{window}d"] = safe_div(
        creative_day[f"clicks_prev_{window}d"], creative_day[f"impressions_prev_{window}d"]
    )
    creative_day[f"cvr_prev_{window}d"] = safe_div(
        creative_day[f"conversions_prev_{window}d"], creative_day[f"clicks_prev_{window}d"]
    )
    creative_day[f"ipm_prev_{window}d"] = safe_div(
        creative_day[f"conversions_prev_{window}d"] * 1000, creative_day[f"impressions_prev_{window}d"]
    )
    creative_day[f"roas_prev_{window}d"] = safe_div(
        creative_day[f"revenue_usd_prev_{window}d"], creative_day[f"spend_usd_prev_{window}d"]
    )

creative_day.head()


,creative_id,campaign_id,date,days_since_launch,spend_usd,impressions,viewable_impressions,clicks,conversions,revenue_usd,video_completions,country_os_slices,ctr,cvr,ipm,roas,viewability_rate,video_completion_rate,impressions_prev_3d,clicks_prev_3d,conversions_prev_3d,spend_usd_prev_3d,revenue_usd_prev_3d,ctr_prev_3d,cvr_prev_3d,ipm_prev_3d,roas_prev_3d,impressions_prev_7d,clicks_prev_7d,conversions_prev_7d,spend_usd_prev_7d,revenue_usd_prev_7d,ctr_prev_7d,cvr_prev_7d,ipm_prev_7d,roas_prev_7d,impressions_prev_14d,clicks_prev_14d,conversions_prev_14d,spend_usd_prev_14d,revenue_usd_prev_14d,ctr_prev_14d,cvr_prev_14d,ipm_prev_14d,roas_prev_14d
0,500000,20000,2026-01-19,0,5917.42,940460,769746,10626,2156,11713.79,0,8,0.011299,0.202899,2.292495,1.979543,0.818478,0.0,0.0,0.0,0.0,0.00,0.00,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.00,0.00,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.00,0.00,NaN,NaN,NaN,NaN
1,500000,20000,2026-01-20,1,2669.38,450862,376573,4302,905,4990.81,0,8,0.009542,0.210367,2.007266,1.869651,0.835229,0.0,940460.0,10626.0,2156.0,5917.42,11713.79,0.011299,0.202899,2.292495,1.979543,940460.0,10626.0,2156.0,5917.42,11713.79,0.011299,0.202899,2.292495,1.979543,940460.0,10626.0,2156.0,5917.42,11713.79,0.011299,0.202899,2.292495,1.979543
2,500000,20000,2026-01-21,2,2804.97,455904,381052,3924,721,4024.43,0,8,0.008607,0.183741,1.581473,1.434750,0.835816,0.0,1391322.0,14928.0,3061.0,8586.80,16704.60,0.010729,0.205051,2.200066,1.945381,1391322.0,14928.0,3061.0,8586.80,16704.60,0.010729,0.205051,2.200066,1.945381,1391322.0,14928.0,3061.0,8586.80,16704.60,0.010729,0.205051,2.200066,1.945381
3,500000,20000,2026-01-22,3,2202.00,347661,283529,2831,550,2832.93,0,8,0.008143,0.194278,1.582001,1.286526,0.815533,0.0,1847226.0,18852.0,3782.0,11391.77,20729.03,0.010206,0.200615,2.047394,1.819650,1847226.0,18852.0,3782.0,11391.77,20729.03,0.010206,0.200615,2.047394,1.819650,1847226.0,18852.0,3782.0,11391.77,20729.03,0.010206,0.200615,2.047394,1.819650
4,500000,20000,2026-01-23,4,2332.58,380006,303292,2475,467,2374.40,0,8,0.006513,0.188687,1.228928,1.017929,0.798124,0.0,1254427.0,11057.0,2176.0,7676.35,11848.17,0.008814,0.196798,1.734657,1.543464,2194887.0,21683.0,4332.0,13593.77,23561.96,0.009879,0.199788,1.973678,1.733291,2194887.0,21683.0,4332.0,13593.77,23561.96,0.009879,0.199788,1.973678,1.733291


## 4. Summaries como targets/agregados

`creative_summary.csv` y `campaign_summary.csv` estan validados contra la fact table. Aun asi, para prediccion temprana pueden introducir leakage porque contienen metricas de todo el periodo. Por eso se separan en tablas de targets/KPIs.

In [9]:
creative_target_cols = [
    "creative_id",
    "creative_status",
    "fatigue_day",
    "perf_score",
    "overall_ctr",
    "overall_cvr",
    "overall_ipm",
    "overall_roas",
    "ctr_decay_pct",
    "cvr_decay_pct",
    "peak_rolling_ctr_5",
]

creative_targets = creative_summary[creative_target_cols].copy()
creative_targets["fatigue_day"] = pd.to_numeric(creative_targets["fatigue_day"], errors="coerce").astype("Int64")
creative_targets["has_fatigue"] = creative_targets["fatigue_day"].notna().astype("int8")

campaign_kpi_cols = [
    "campaign_id",
    "total_spend_usd",
    "total_impressions",
    "total_clicks",
    "total_conversions",
    "total_revenue_usd",
    "overall_ctr",
    "overall_cvr",
    "overall_roas",
]
campaign_kpis = campaign_summary[campaign_kpi_cols].copy()

creative_targets.shape, campaign_kpis.shape


((1080, 12), (180, 9))

## 5. Starter dataset a nivel creative_id

Esta tabla es comoda para un primer modelo por creatividad. Las columnas `first_*` se calculan desde `daily`; las columnas de target se agregan al final y deben separarse antes de entrenar segun el caso de uso.

In [10]:
def aggregate_window(day_df: pd.DataFrame, max_day_inclusive: int, prefix: str) -> pd.DataFrame:
    window_df = day_df.loc[day_df["days_since_launch"].le(max_day_inclusive)].copy()
    out = (
        window_df.groupby("creative_id", as_index=False)
        .agg(
            **{
                f"{prefix}_days_observed": ("date", "nunique"),
                f"{prefix}_spend_usd": ("spend_usd", "sum"),
                f"{prefix}_impressions": ("impressions", "sum"),
                f"{prefix}_viewable_impressions": ("viewable_impressions", "sum"),
                f"{prefix}_clicks": ("clicks", "sum"),
                f"{prefix}_conversions": ("conversions", "sum"),
                f"{prefix}_revenue_usd": ("revenue_usd", "sum"),
                f"{prefix}_avg_country_os_slices": ("country_os_slices", "mean"),
                f"{prefix}_ctr_volatility": ("ctr", "std"),
                f"{prefix}_ipm_volatility": ("ipm", "std"),
            }
        )
    )
    out[f"{prefix}_ctr"] = safe_div(out[f"{prefix}_clicks"], out[f"{prefix}_impressions"])
    out[f"{prefix}_viewability_rate"] = safe_div(out[f"{prefix}_viewable_impressions"], out[f"{prefix}_impressions"])
    out[f"{prefix}_cvr"] = safe_div(out[f"{prefix}_conversions"], out[f"{prefix}_clicks"])
    out[f"{prefix}_ipm"] = safe_div(out[f"{prefix}_conversions"] * 1000, out[f"{prefix}_impressions"])
    out[f"{prefix}_roas"] = safe_div(out[f"{prefix}_revenue_usd"], out[f"{prefix}_spend_usd"])
    return out


first_3d = aggregate_window(creative_day, 2, "first_3d")
first_7d = aggregate_window(creative_day, 6, "first_7d")
first_14d = aggregate_window(creative_day, 13, "first_14d")

full_lifecycle = (
    creative_day.groupby("creative_id", as_index=False)
    .agg(
        lifecycle_days=("date", "nunique"),
        lifecycle_spend_usd=("spend_usd", "sum"),
        lifecycle_impressions=("impressions", "sum"),
        lifecycle_clicks=("clicks", "sum"),
        lifecycle_conversions=("conversions", "sum"),
        lifecycle_revenue_usd=("revenue_usd", "sum"),
        lifecycle_ctr_std=("ctr", "std"),
        lifecycle_ipm_std=("ipm", "std"),
        lifecycle_max_days_since_launch=("days_since_launch", "max"),
    )
)
full_lifecycle["lifecycle_ctr"] = safe_div(full_lifecycle["lifecycle_clicks"], full_lifecycle["lifecycle_impressions"])
full_lifecycle["lifecycle_cvr"] = safe_div(full_lifecycle["lifecycle_conversions"], full_lifecycle["lifecycle_clicks"])
full_lifecycle["lifecycle_ipm"] = safe_div(full_lifecycle["lifecycle_conversions"] * 1000, full_lifecycle["lifecycle_impressions"])
full_lifecycle["lifecycle_roas"] = safe_div(full_lifecycle["lifecycle_revenue_usd"], full_lifecycle["lifecycle_spend_usd"])

creative_feature_base = (
    creative_dim
    .merge(first_3d, on="creative_id", how="left", validate="1:1")
    .merge(first_7d, on="creative_id", how="left", validate="1:1")
    .merge(first_14d, on="creative_id", how="left", validate="1:1")
    .merge(full_lifecycle, on="creative_id", how="left", validate="1:1")
    .merge(creative_targets, on="creative_id", how="left", validate="1:1")
)

creative_feature_base.shape


(1080, 119)

In [11]:
TARGET_COLUMNS = creative_target_cols[1:] + ["has_fatigue"]
LEAKAGE_PRONE_COLUMNS = [c for c in creative_feature_base.columns if c.startswith("lifecycle_")] + TARGET_COLUMNS
EARLY_FEATURE_COLUMNS = [
    c for c in creative_feature_base.columns
    if c not in LEAKAGE_PRONE_COLUMNS and not c.startswith("total_") and not c.startswith("overall_")
]

column_roles = pd.DataFrame(
    [
        {"column": c, "role": "target_or_label" if c in TARGET_COLUMNS else "full_period_metric" if c.startswith("lifecycle_") else "candidate_feature"}
        for c in creative_feature_base.columns
    ]
)

column_roles["role"].value_counts().rename_axis("role").reset_index(name="columns")


,role,columns
0,candidate_feature,95
1,full_period_metric,13
2,target_or_label,11


## 6. Guardado

Al ejecutar esta celda se escriben CSVs procesados en `feature/processed/`. No se sobrescriben los CSV originales.

In [12]:
outputs = {
    "advertisers_clean": advertisers_clean,
    "campaign_dim": campaign_dim,
    "campaign_country": campaign_country,
    "creative_dim": creative_dim,
    "daily_enriched": daily_enriched,
    "creative_day": creative_day,
    "creative_targets": creative_targets,
    "campaign_kpis": campaign_kpis,
    "creative_feature_base": creative_feature_base,
    "column_roles": column_roles,
}

for name, df in outputs.items():
    df.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)

preprocessing_report = pd.DataFrame(
    [
        {"object": name, "rows": len(df), "columns": df.shape[1], "file": str(OUTPUT_DIR / f"{name}.csv")}
        for name, df in outputs.items()
    ]
)
preprocessing_report.to_csv(OUTPUT_DIR / "preprocessing_report.csv", index=False)
preprocessing_report


,object,rows,columns,file
0,advertisers_clean,36,4,/home/wei/projects/hack_UPC_2026/feature/proce...
1,campaign_dim,180,15,/home/wei/projects/hack_UPC_2026/feature/proce...
2,campaign_country,502,2,/home/wei/projects/hack_UPC_2026/feature/proce...
3,creative_dim,1080,50,/home/wei/projects/hack_UPC_2026/feature/proce...
4,daily_enriched,192315,67,/home/wei/projects/hack_UPC_2026/feature/proce...
5,creative_day,52349,45,/home/wei/projects/hack_UPC_2026/feature/proce...
6,creative_targets,1080,12,/home/wei/projects/hack_UPC_2026/feature/proce...
7,campaign_kpis,180,9,/home/wei/projects/hack_UPC_2026/feature/proce...
8,creative_feature_base,1080,119,/home/wei/projects/hack_UPC_2026/feature/proce...
9,column_roles,119,2,/home/wei/projects/hack_UPC_2026/feature/proce...
